# Lab 2

## Task 1 - Look at the data before you model it
We start by setting up the seed and the hardware configuration, then we load json files and extract their features. 
After this, we extract a stratified validation set from the `train.json` since we cannot use `test.json`. With the option 
```
stratify=y_train_full
```
we are enforcing the stratification to keep the same ratio of goodware/malware in both the splits.
To answer the question "does the order of API calls in sequences really matters for malware identification?" we create the `shuffle_within` function that takes sequences of `test.json` and shuffles them randomly

In [ ]:
import json
import random
import numpy as np
import torch
from sklearn.model_selection import train_test_split

# =====================================================================
# 1. Seed & Device Setup
# =====================================================================
SEED = 42

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

# =====================================================================
# 2. Load JSON Datasets & Exact Train/Val/Test Split
# =====================================================================
with open("train.json", "r") as f:
    train_raw = json.load(f)
with open("test.json", "r") as f:
    test_raw = json.load(f)

X_train_full = [x["api_call_sequence"] for x in train_raw]
y_train_full = np.array([x["is_malware"] for x in train_raw])
X_test = [x["api_call_sequence"] for x in test_raw]
y_test = np.array([x["is_malware"] for x in test_raw])

# ==========================================================
# Exact same 80/20 stratified split as the baseline notebook
# ==========================================================
indices = np.arange(len(X_train_full))
train_idx, val_idx = train_test_split(
    indices,
    test_size=0.2,
    stratify=y_train_full,
    random_state=SEED,
)

X_train = [X_train_full[i] for i in train_idx]
y_train = y_train_full[train_idx]
X_val = [X_train_full[i] for i in val_idx]
y_val = y_train_full[val_idx]

# ==================================================================
# Shuffled test set (randomizes call order within each sequence)
# ==================================================================
def shuffle_within(sequences, seed=SEED):
    rng = random.Random(seed)
    return [rng.sample(s, len(s)) for s in sequences]

X_test_sh = shuffle_within(X_test)

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    classification_report,
    confusion_matrix
)

# =====================================================================
# 3. Task 1.1: Class Balance
# =====================================================================
def print_class_balance(y, name):
    total = len(y)
    malware = np.sum(y == 1)
    goodware = np.sum(y == 0)
    mal_pct = (malware / total) * 100
    good_pct = (goodware / total) * 100
    print(f"--- {name} (Totale: {total}) ---")
    print(f"Malware (1):  {malware} ({mal_pct:.2f}%)")
    print(f"Goodware (0): {goodware} ({good_pct:.2f}%)")
    print(f"Dummy Majority Classifier Accuracy: {max(mal_pct, good_pct):.2f}%\n")

print_class_balance(y_train_full, "Full Train Set (train.json)")
print_class_balance(y_train, "Train Split (80%)")
print_class_balance(y_val, "Validation Split (20%)")
print_class_balance(y_test, "Test Set (test.json)")

# =====================================================================
# 6. Funzione di Valutazione Universale (per Task 1 -> Task 6)
# =====================================================================
def evaluate_predictions(y_true, y_pred, dataset_name="Validation"):
    """
    Calcola e stampa le metriche scelte nel Task 1:
    - Macro-F1 (metrica primaria per la model selection)
    - F1, Recall e Precision separate per Goodware (0) e Malware (1)
    - Accuracy (mostrata solo per confronto)
    """
    acc = accuracy_score(y_true, y_pred)
    macro_f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)
    
    # Metriche per singola classe: indice 0 = Goodware, indice 1 = Malware
    f1_per_class = f1_score(y_true, y_pred, average=None, labels=[0, 1], zero_division=0)
    rec_per_class = recall_score(y_true, y_pred, average=None, labels=[0, 1], zero_division=0)
    prec_per_class = precision_score(y_true, y_pred, average=None, labels=[0, 1], zero_division=0)
    
    metrics = {
        "accuracy": acc,
        "macro_f1": macro_f1,
        "goodware_f1": f1_per_class[0],
        "goodware_recall": rec_per_class[0],
        "goodware_precision": prec_per_class[0],
        "malware_f1": f1_per_class[1],
        "malware_recall": rec_per_class[1],
        "malware_precision": prec_per_class[1],
    }
    
    print(f"=== Risultati su {dataset_name} ===")
    print(f"Macro-F1 (Metrica Primaria): {macro_f1:.4f}  |  (Accuracy: {acc:.4f})")
    print(f"  -> Goodware (0 - Minoritaria): F1={f1_per_class[0]:.4f} | "
          f"Recall={rec_per_class[0]:.4f} | Precision={prec_per_class[0]:.4f}")
    print(f"  -> Malware  (1 - Maggioritaria): F1={f1_per_class[1]:.4f} | "
          f"Recall={rec_per_class[1]:.4f} | Precision={prec_per_class[1]:.4f}\n")
    
    return metrics

# =====================================================================
# 7. Dimostrazione Task 1.1: Cosa succede col Dummy Majority Classifier?
# =====================================================================
# Creiamo un vettore di predizioni che dice sempre 1 (Malware) sul Validation Set
y_val_dummy = np.ones_like(y_val)
print("--- DUMMY CLASSIFIER (Predice sempre Malware=1) ---")
dummy_metrics = evaluate_predictions(y_val, y_val_dummy, dataset_name="Validation (Dummy)")


In [ ]:
import matplotlib.pyplot as plt

# =====================================================================
# 4. Task 1.2: Sequence Lengths Distribution
# =====================================================================
train_lengths = np.array([len(seq) for seq in X_train_full])
test_lengths = np.array([len(seq) for seq in X_test])

def print_length_stats(lengths, name):
    print(f"--- Statistiche Lunghezza Sequenze: {name} ---")
    print(f"Min:            {np.min(lengths)}")
    print(f"Max:            {np.max(lengths)}")
    print(f"Media (Mean):   {np.mean(lengths):.2f} (Std: {np.std(lengths):.2f})")
    print(f"Mediana (p50):  {np.median(lengths):.1f}")
    print(f"Percentili:     p25={np.percentile(lengths, 25):.1f}, "
          f"p75={np.percentile(lengths, 75):.1f}, "
          f"p90={np.percentile(lengths, 90):.1f}, "
          f"p95={np.percentile(lengths, 95):.1f}, "
          f"p99={np.percentile(lengths, 99):.1f}\n")

print_length_stats(train_lengths, "Train (Full)")
print_length_stats(test_lengths, "Test")

# Plot delle distribuzioni (Scala normale e logaritmica per code lunghe)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(train_lengths, bins=50, alpha=0.6, label="Train", density=True)
axes[0].hist(test_lengths, bins=50, alpha=0.6, label="Test", density=True)
axes[0].set_title("Distribuzione Lunghezza Sequenze (Densità)")
axes[0].set_xlabel("Lunghezza sequenza (# API calls)")
axes[0].set_ylabel("Densità")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].hist(train_lengths, bins=50, alpha=0.6, label="Train")
axes[1].hist(test_lengths, bins=50, alpha=0.6, label="Test")
axes[1].set_yscale("log")
axes[1].set_title("Distribuzione Lunghezza Sequenze (Count - Log Scale)")
axes[1].set_xlabel("Lunghezza sequenza (# API calls)")
axes[1].set_ylabel("Frequenza (scala log)")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# =====================================================================
# 5. Task 1.3: Vocabulary & Out-of-Vocabulary (OOV) Analysis
# =====================================================================
# NOTA: La traccia richiede di usare SOLO la partizione di training (X_train)
vocab_train = set(call for seq in X_train for call in seq)
vocab_test = set(call for seq in X_test for call in seq)

# Chiamate presenti nel test set ma mai viste in X_train
oov_calls = vocab_test - vocab_train

# Campioni del test set che contengono almeno una chiamata OOV
test_samples_with_oov = sum(
    1 for seq in X_test if any(call in oov_calls for call in seq)
)

print("--- Analisi Vocabolario (basato su X_train post-split) ---")
print(f"API call distinte in X_train (Vocab size): {len(vocab_train)}")
print(f"API call distinte in X_test:               {len(vocab_test)}")
print(f"API call OOV distinte nel test set:        {len(oov_calls)}")
if oov_calls:
    print(f"Elenco chiamate OOV:                       {sorted(list(oov_calls))}")
print(f"Campioni di test con almeno 1 OOV:         {test_samples_with_oov} / {len(X_test)} "
      f"({(test_samples_with_oov / len(X_test)) * 100:.2f}%)")

In [ ]:
from sklearn.utils.class_weight import compute_class_weight

# =====================================================================
# 8. Calcolo dei Class Weights (stimati SOLO su y_train, per Task 2-5)
# =====================================================================
# Formula bilanciata: n_samples / (n_classes * np.bincount(y_train))
class_weights_np = compute_class_weight(
    class_weight="balanced",
    classes=np.array([0, 1]),
    y=y_train
)
class_weights_tensor = torch.tensor(class_weights_np, dtype=torch.float32).to(device)

print("--- Pesi delle Classi (calcolati esclusivamente su y_train) ---")
print(f"Peso classe 0 (Goodware): {class_weights_np[0]:.4f}")
print(f"Peso classe 1 (Malware):  {class_weights_np[1]:.4f}")

### Riepilogo 
Riepilogo completo del Task 1 (Pronto per il tuo Report)
1. Class Balance & Metriche: Forte sbilanciamento con 96,26% Malware e 3,74% Goodware costante in tutte le partizioni. Un dummy classifier che predice sempre Malware ottiene il 96,26% di Accuracy, rendendo l'Accuracy inutilizzabile. Useremo la Macro-F1 come metrica principale (insieme a F1/Recall sulla classe minoritaria Goodware e Recall su Malware) e peseremo la funzione di costo sui pesi bilanciati calcolati su y_train.
2. Sequence Lengths: Le lunghezze in Train sono distribuite uniformemente tra 60 e 90 (media 75,03, mediana 75,0), mentre nel Test set subiscono uno shift verso valori più alti, tra 70 e 100 (media 86,33, mediana 87,0, con oltre il 25% dei sample più lunghi di 90). Per la FFNN del Task 3, fissando $L = 90$ (il massimo del training set), dovremo fare padding per le sequenze $< 90$ e truncation alle prime 90 chiamate per i campioni del test set che hanno lunghezza tra 91 e 100.
3. Vocabulary & OOV: 256 chiamate distinte in X_train, 3 chiamate OOV nel test set (ControlService, NtDeleteKey, WSASocketA) distribuite su 3 campioni (0,05% del test set).  

# Task 2

In [ ]:
import time
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression

# =====================================================================
# TASK 2A: Unigram Bag-of-Words (Order-Blind Baseline)
# =====================================================================

# Funzione identità per permettere a CountVectorizer di leggere liste di stringhe
def identity_tokenizer(seq):
    return seq

# 1. Fit SOLO su X_train (13.060 sequenze), Transform su Val e Test
vec_unigram = CountVectorizer(
    tokenizer=identity_tokenizer,
    preprocessor=identity_tokenizer,
    token_pattern=None,
    lowercase=False,
    ngram_range=(1, 1)  # Solo unigrammi (singole API call)
)

X_train_uni = vec_unigram.fit_transform(X_train)
X_val_uni = vec_unigram.transform(X_val)
X_test_uni = vec_unigram.transform(X_test)
X_test_sh_uni = vec_unigram.transform(X_test_sh)

print("--- Verifica Rappresentazione Unigram BoW ---")
print(f"Shape X_train_uni: {X_train_uni.shape} (Righe=Sample, Colonne=Vocab size)")
print(f"Shape X_val_uni:   {X_val_uni.shape}")
print(f"Shape X_test_uni:  {X_test_uni.shape}")

# Verifica formale (Domanda Sezione 4.4 / Task 2B):
# Lo shuffle delle sequenze cambia i vettori Unigram? (Atteso: 0 differenze)
diff_sh_uni = (X_test_uni != X_test_sh_uni).nnz
print(f"Elementi diversi tra X_test_uni e X_test_sh_uni dopo shuffle: {diff_sh_uni}\n")

# =====================================================================
# 2. Esperimenti e Tuning su Validation Set (Logbook: Pesi e parametro C)
# =====================================================================
# Confrontiamo senza class_weight vs con class_weight='balanced' e diversi valori di C
configs_2a = [
    {"C": 1.0, "class_weight": None,       "desc": "LR (C=1.0, No Class Weights)"},
    {"C": 0.1, "class_weight": "balanced", "desc": "LR (C=0.1, Balanced Weights)"},
    {"C": 1.0, "class_weight": "balanced", "desc": "LR (C=1.0, Balanced Weights)"},
    {"C": 10.0, "class_weight": "balanced", "desc": "LR (C=10.0, Balanced Weights)"},
]

best_val_f1_uni = -1.0
best_model_uni = None
best_cfg_uni = None
best_time_uni = 0.0

print("--- Tuning Logistic Regression su Unigram BoW (Validation Set) ---")
for cfg in configs_2a:
    t0 = time.time()
    clf = LogisticRegression(
        C=cfg["C"],
        class_weight=cfg["class_weight"],
        max_iter=10000,
        random_state=SEED
    )
    clf.fit(X_train_uni, y_train)
    train_time = time.time() - t0
    
    y_val_pred = clf.predict(X_val_uni)
    print(f"Config: {cfg['desc']} (Tempo train: {train_time:.3f}s)")
    metrics = evaluate_predictions(y_val, y_val_pred, dataset_name=f"Val - {cfg['desc']}")
    
    if metrics["macro_f1"] > best_val_f1_uni:
        best_val_f1_uni = metrics["macro_f1"]
        best_model_uni = clf
        best_cfg_uni = cfg
        best_time_uni = train_time

print("=====================================================================")
print(f"MIGLIOR MODELLO 2A (selezionato su Validation): {best_cfg_uni['desc']}")
print(f"Validation Macro-F1: {best_val_f1_uni:.4f} | Training Time: {best_time_uni:.3f}s")
print("=====================================================================\n")

### Task 2A
##### Q1: Quante colonne produce il vectorizer e perché? Cosa rappresenta ogni riga? Puoi recuperare l'ordine originale?
- Produce esattamente 256 colonne, perché nel Task 1 abbiamo visto che ci sono esattamente 256 API call distinte nella partizione X_train su cui viene fatto il .fit(). 
- Ogni riga rappresenta un singolo programma (campione) codificato come il vettore dei conteggi (frequenze assolute) di ciascuna delle 256 API call all'interno della sua esecuzione.
- No, è impossibile recuperare l'ordine originale: la rappresentazione Bag-of-Words è un multinsieme (order-blind). Sapere che NtClose compare 5 volte e LdrLoadDll 3 volte non ci dice in quale posizione temporale siano avvenute.

##### Q2: Fai un esempio di due sequenze diverse che producono lo stesso vettore unigram. Quale informazione si perde? Due sequenze con lo stesso insieme di API call possono produrre vettori unigram diversi? Fai un esempio
- Stesso vettore da sequenze diverse: La sequenza $S_1 =$ ["LdrLoadDll", "NtClose"] e la sequenza $S_2 =$ ["NtClose", "LdrLoadDll"] sono diverse ma producono esattamente lo stesso vettore unigram (un 1 nella colonna di LdrLoadDll, un 1 nella colonna di NtClose, 0 altrove.
- L'informazione persa è l'ordine sequenziale e le transizioni tra chiamate.
- Vettori diversi dallo stesso insieme (set) di chiamate: Sì, perché il CountVectorizer conta le frequenze (molteplicità) e non solo la presenza/assenza binaria (ricordando che nel dataset sono state collassate solo le chiamate identiche consecutive, ma una chiamata può riapparire più avanti nella sequenza!). Ad esempio, $S_A =$ ["LdrLoadDll", "NtClose"] e $S_B =$ ["LdrLoadDll", "NtClose", "LdrLoadDll"] contengono esattamente lo stesso insieme di API distinte $\{ \text{LdrLoadDll}, \text{NtClose} \}$, ma in $S_A$ il conteggio di LdrLoadDll è 1, mentre in $S_B$ è 2, generando quindi due vettori diversi.

##### Q3: Out-of-vocabulary (OOV): cosa fa CountVectorizer a test time con le chiamate mai viste in training?
Durante .transform(X_test), CountVectorizer non solleva alcun errore, ma ignora silenziosamente qualsiasi chiamata non presente nelle 256 parole del vocabolario appreso su X_train. Come calcolato nel Task 1, questo ignora le 3 chiamate OOV (ControlService, NtDeleteKey, WSASocketA) impattando appena 3 campioni su 6.505 (0,05%) del test set.

##### Q4: Ipotesi pre-training (per il Logbook - Run 1)
Ipotesi: Ci aspettiamo che anche un modello order-blind basato sulle sole frequenze (es. Logistic Regression) ottenga una buona Macro-F1, perché certi malware utilizzano specifiche API di sistema (es. iniezione in memoria, manipolazione registro/servizi) con frequenze molto diverse dal software benigno. Tuttavia, senza class_weight="balanced", la Recall sulla classe rara (Goodware) rischia di essere più bassa; applicando il bilanciamento dei pesi e ottimizzando la regolarizzazione $C$, ci aspettiamo un forte recupero della Recall sul Goodware (anche a costo di qualche falso negativo in più sul Malware).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

# =====================================================================
# Grafici Task 2A: Confronto Metriche e Matrici di Confusione
# =====================================================================
# Raccogliamo le metriche e le predizioni per il plot
results_2a = {}
preds_2a = {}

for cfg in configs_2a:
    clf = LogisticRegression(
        C=cfg["C"],
        class_weight=cfg["class_weight"],
        max_iter=10000,
        random_state=SEED
    )
    clf.fit(X_train_uni, y_train)
    y_p = clf.predict(X_val_uni)
    preds_2a[cfg["desc"]] = y_p
    results_2a[cfg["desc"]] = {
        "Macro-F1": f1_score(y_val, y_p, average="macro"),
        "Goodware F1": f1_score(y_val, y_p, pos_label=0),
        "Goodware Recall": recall_score(y_val, y_p, pos_label=0),
        "Goodware Precision": precision_score(y_val, y_p, pos_label=0),
        "Malware Recall": recall_score(y_val, y_p, pos_label=1),
    }

# 1. Grafico a barre raggruppate: Trade-off delle metriche al variare dei pesi e di C
labels = list(results_2a.keys())
metric_names = ["Macro-F1", "Goodware F1", "Goodware Recall", "Goodware Precision", "Malware Recall"]
x = np.arange(len(labels))
width = 0.15

fig, ax = plt.subplots(figsize=(13, 5))
for i, m_name in enumerate(metric_names):
    vals = [results_2a[lbl][m_name] for lbl in labels]
    rects = ax.bar(x + (i - 2) * width, vals, width, label=m_name)
    ax.bar_label(rects, fmt="%.2f", padding=2, fontsize=8)

ax.set_ylabel("Score (0.0 - 1.0)")
ax.set_title("Task 2A (Unigram BoW): Effetto di Class Weights e Regolarizzazione C su Validation Set")
ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=10)
ax.set_ylim(0, 1.12)
ax.legend(loc="upper right", ncol=3)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

# 2. Matrici di Confusione affiancate: No Weights (C=1.0) vs Balanced Weights (C=1.0)
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

compare_keys = ["LR (C=1.0, No Class Weights)", "LR (C=1.0, Balanced Weights)"]
for idx, key in enumerate(compare_keys):
    cm = confusion_matrix(y_val, preds_2a[key], labels=[0, 1])
    sns.heatmap(
        cm, annot=True, fmt="d", cmap="Blues", ax=axes[idx],
        xticklabels=["Pred: Goodware (0)", "Pred: Malware (1)"],
        yticklabels=["True: Goodware (0)", "True: Malware (1)"]
    )
    axes[idx].set_title(f"Confusion Matrix: {key}")

plt.tight_layout()
plt.show()

- Senza pesi (No Class Weights, $C=1.0$): Il modello è conservativo sulla classe rara. Individua solo il 31,15% dei Goodware (Recall = 0.3115), ma quando dice che un programma è Goodware ha ragione il 67,86% delle volte (Precision = 0.6786). La F1 sul Goodware è 0.4270 e la Macro-F1 arriva a 0.7055.
- Con pesi bilanciati (Balanced Weights, $C=1.0$): Poiché sbagliare un Goodware "costa" 25,7 volte più che sbagliare un Malware, l'iperpiano della Logistic Regression si sposta drasticamente per catturare più Goodware possibili. La Goodware Recall balza dal 31,15% al 76,23% (+45 punti percentuali!).Tuttavia, nello spazio delle sole frequenze unigram (256 colonne), molti Malware condividono frequenze di API simili ai Goodware. Per catturare quel 76% di Goodware, il modello finisce per classificare erroneamente circa il 12% dei Malware (quasi 370 malware nel validation set!) come Goodware (Malware Recall scende da 0.9943 a 0.8810).   Essendoci 3.143 Malware contro soli 122 Goodware nel validation set, quel 12% di Malware sbagliati inonda la classe Goodware di falsi Goodware, facendo crollare la Goodware Precision al 19,91% (0.1991) e trascinando giù la F1 del Goodware (0.3158) e la Macro-F1 (0.6240)
- *Implicazione pratica per un Malware Detector* (Task 1.1 & 2A): In un antivirus reale, avere una Malware Recall che scende all'88,1% (lasciando passare quasi il 12% dei malware come software benigno!) è molto pericoloso. Puoi anche testare un peso intermedio (es. class_weight={0: 3.0, 1: 1.0} o {0: 5.0, 1: 1.0}) se vuoi vedere se esiste un compromesso migliore tra None (rapporto 1:1) e balanced (rapporto 25.7:1)!  

### Task 2B
Il Task 2B (Bigram BoW) aggiunge alla rappresentazione unigram anche le coppie di chiamate API immediatamente adiacenti configurando CountVectorizer con ngram_range=(1, 2). In questo modo il modello non conosce solo quante volte viene invocata una singola API, ma cattura anche l'ordine locale (transizioni dirette del tipo $A \rightarrow B$).
#### 1. Risposte alle domande teoriche del Task 2B (per il Report e il Logbook)
##### Ipotesi pre-training (Domanda 1 & Logbook):Ci aspettiamo un miglioramento?
Sì. Dal punto di vista del comportamento del software (e in particolare del malware), una singola API call isolata come VirtualAllocEx, WriteProcessMemory o RegOpenKeyExW può apparire sia in programmi benigni sia in malware. Tuttavia, la sequenza immediata di due chiamate consecutive (es. aprire un processo/chiave di registro e subito dopo scriverci dentro o allocare memoria eseguibile per una process injection) costituisce una micro-firma comportamentale molto più specifica e discriminante tra malware e goodware.
##### Cosa potrebbe invece ostacolare il miglioramento?
Se l'aggiunta dei bigrammi fa esplodere il numero di colonne (alta dimensionalità e sparsità), un modello lineare poco regolarizzato potrebbe soffrire di overfitting sulle coppie rare viste solo nel training set.
##### Effetto dello Shuffle Test su Unigram vs Bigram (Domanda 3 & Sezione 4.4):Perché i vettori Unigram NON cambiano dopo lo shuffle? 
Perché rimescolare una sequenza s con rng.sample(s, len(s)) cambia la posizione degli elementi ma lascia identici i conteggi delle singole chiamate. Di conseguenza X_test_uni e X_test_sh_uni sono matematicamente identici (0 elementi diversi, come hai appena verificato!) e il modello Unigram darà esattamente le stesse predizioni e lo stesso punteggio prima e dopo lo shuffle.  
##### Perché i vettori Bigram CAMBIANO dopo lo shuffle? 
Rimescolando casualmente le chiamate dentro una sequenza, le coppie adiacenti originali $(c_t, c_{t+1})$ vengono spezzate e si formano nuove coppie casuali che non riflettono più il vero flusso di esecuzione del programma.
##### Cosa dimostra il crollo (o la tenuta) delle performance sul Test Set Shuffled? 
Se il modello Bigram ottiene una Macro-F1 più alta dell'Unigram sul test set originale, ma peggiora sensibilmente quando viene valutato su X_test_sh (senza riaddestrarlo), avrai la prova sperimentale che il modello Bigram sta effettivamente sfruttando l'ordine locale delle chiamate API per classificare i campioni.

In [ ]:
# =====================================================================
# TASK 2B: Bigram Bag-of-Words (Unigrams + Bigrams, Local Order)
# =====================================================================

# 1. Fit SOLO su X_train con ngram_range=(1, 2)
# Nota: in CountVectorizer i token vengono uniti con uno spazio se l'input è stringa;
# passando direttamente le liste di stringhe tramite un analyzer custom generiamo
# sia i singoli token (unigrammi) sia le coppie adiacenti (bigrammi).
def unigram_bigram_analyzer(seq):
    unigrams = list(seq)
    bigrams = [f"{seq[i]} -> {seq[i+1]}" for i in range(len(seq) - 1)]
    return unigrams + bigrams

vec_bigram = CountVectorizer(
    analyzer=unigram_bigram_analyzer
)

X_train_bi = vec_bigram.fit_transform(X_train)
X_val_bi = vec_bigram.transform(X_val)
X_test_bi = vec_bigram.transform(X_test)
X_test_sh_bi = vec_bigram.transform(X_test_sh)

print("--- Verifica Rappresentazione Bigram BoW (1, 2) ---")
print(f"Shape X_train_bi: {X_train_bi.shape} "
      f"(256 unigrammi + {X_train_bi.shape[1] - 256} bigrammi distinti)")
print(f"Shape X_val_bi:   {X_val_bi.shape}")
print(f"Shape X_test_bi:  {X_test_bi.shape}")

# Verifica formale: lo shuffle cambia i vettori Bigram? (Atteso: > 0 differenze)
diff_sh_bi = (X_test_bi != X_test_sh_bi).nnz
print(f"Elementi diversi tra X_test_bi e X_test_sh_bi dopo shuffle: {diff_sh_bi}\n")

# =====================================================================
# 2. Esperimenti e Tuning su Validation Set (Stesse config di 2A)
# =====================================================================
configs_2b = [
    {"C": 1.0,  "class_weight": None,       "desc": "Bigram LR (C=1.0, No Weights)"},
    {"C": 0.1,  "class_weight": "balanced", "desc": "Bigram LR (C=0.1, Balanced)"},
    {"C": 1.0,  "class_weight": "balanced", "desc": "Bigram LR (C=1.0, Balanced)"},
    {"C": 10.0, "class_weight": "balanced", "desc": "Bigram LR (C=10.0, Balanced)"},
]

best_val_f1_bi = -1.0
best_model_bi = None
best_cfg_bi = None
best_time_bi = 0.0
results_2b = {}

print("--- Tuning Logistic Regression su Bigram BoW (Validation Set) ---")
for cfg in configs_2b:
    t0 = time.time()
    clf = LogisticRegression(
        C=cfg["C"],
        class_weight=cfg["class_weight"],
        max_iter=10000,
        random_state=SEED
    )
    clf.fit(X_train_bi, y_train)
    train_time = time.time() - t0
    
    y_val_pred = clf.predict(X_val_bi)
    print(f"Config: {cfg['desc']} (Tempo train: {train_time:.3f}s)")
    metrics = evaluate_predictions(y_val, y_val_pred, dataset_name=f"Val - {cfg['desc']}")
    results_2b[cfg["desc"]] = metrics
    
    if metrics["macro_f1"] > best_val_f1_bi:
        best_val_f1_bi = metrics["macro_f1"]
        best_model_bi = clf
        best_cfg_bi = cfg
        best_time_bi = train_time

print("=====================================================================")
print(f"MIGLIOR MODELLO 2B (selezionato su Validation): {best_cfg_bi['desc']}")
print(f"Validation Macro-F1: {best_val_f1_bi:.4f} | Training Time: {best_time_bi:.3f}s")
print("=====================================================================\n")

# =====================================================================
# 3. Shuffle Test sul Test Set (Orig vs Shuffled) per 2A e 2B
# =====================================================================
print("--- SHUFFLE TEST SUL TEST SET (Sezione 4.4 & Task 2B) ---")

# Valutazione Miglior Unigram (2A) su Test Originale e Test Shuffled
test_uni_orig = evaluate_predictions(
    y_test, best_model_uni.predict(X_test_uni),
    dataset_name="TEST ORIGINALE - Best Unigram BoW (2A)"
)
test_uni_shuf = evaluate_predictions(
    y_test, best_model_uni.predict(X_test_sh_uni),
    dataset_name="TEST SHUFFLED  - Best Unigram BoW (2A)"
)

# Valutazione Miglior Bigram (2B) su Test Originale e Test Shuffled
test_bi_orig = evaluate_predictions(
    y_test, best_model_bi.predict(X_test_bi),
    dataset_name="TEST ORIGINALE - Best Bigram BoW (2B)"
)
test_bi_shuf = evaluate_predictions(
    y_test, best_model_bi.predict(X_test_sh_bi),
    dataset_name="TEST SHUFFLED  - Best Bigram BoW (2B)"
)

# Verifica anche quante predizioni individuali cambiano dopo lo shuffle
pred_diff_uni = np.sum(best_model_uni.predict(X_test_uni) != best_model_uni.predict(X_test_sh_uni))
pred_diff_bi  = np.sum(best_model_bi.predict(X_test_bi) != best_model_bi.predict(X_test_sh_bi))
print(f"Predizioni individuali cambiate dopo shuffle (Unigram): {pred_diff_uni} / {len(y_test)}")
print(f"Predizioni individuali cambiate dopo shuffle (Bigram):  {pred_diff_bi} / {len(y_test)}\n")

# =====================================================================
# 4. Grafico Confronto 2A vs 2B (Validation + Shuffle Test)
# =====================================================================
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Plot 1: Confronto Validation Macro-F1 tra 2A e 2B a parità di configurazione
cfg_short_names = ["C=1.0, No Weights", "C=0.1, Balanced", "C=1.0, Balanced", "C=10.0, Balanced"]
val_f1_2a = [results_2a[k]["Macro-F1"] for k in results_2a.keys()]
val_f1_2b = [results_2b[k]["macro_f1"] for k in results_2b.keys()]

x = np.arange(len(cfg_short_names))
width = 0.35
r1 = axes[0].bar(x - width/2, val_f1_2a, width, label="2A: Unigram (1,1)", color="steelblue")
r2 = axes[0].bar(x + width/2, val_f1_2b, width, label="2B: Bigram (1,2)", color="darkorange")
axes[0].bar_label(r1, fmt="%.3f", padding=2, fontsize=8)
axes[0].bar_label(r2, fmt="%.3f", padding=2, fontsize=8)
axes[0].set_title("Validation Set: Unigram (2A) vs Bigram (2B) Macro-F1")
axes[0].set_xticks(x)
axes[0].set_xticklabels(cfg_short_names, rotation=12)
axes[0].set_ylabel("Macro-F1")
axes[0].set_ylim(0, 1.05)
axes[0].legend()
axes[0].grid(axis="y", alpha=0.3)

# Plot 2: Effetto dello Shuffle Test sul Test Set (Original vs Shuffled)
models_lbl = ["Best Unigram (2A)", "Best Bigram (2B)"]
orig_scores = [test_uni_orig["macro_f1"], test_bi_orig["macro_f1"]]
shuf_scores = [test_uni_shuf["macro_f1"], test_bi_shuf["macro_f1"]]

x2 = np.arange(len(models_lbl))
r3 = axes[1].bar(x2 - width/2, orig_scores, width, label="Original Test Set", color="seagreen")
r4 = axes[1].bar(x2 + width/2, shuf_scores, width, label="Shuffled Test Set", color="crimson")
axes[1].bar_label(r3, fmt="%.4f", padding=2, fontsize=9)
axes[1].bar_label(r4, fmt="%.4f", padding=2, fontsize=9)
axes[1].set_title("Order Ablation (Test Set): Original vs Shuffled Macro-F1")
axes[1].set_xticks(x2)
axes[1].set_xticklabels(models_lbl)
axes[1].set_ylabel("Test Macro-F1")
axes[1].set_ylim(0, 1.05)
axes[1].legend()
axes[1].grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

I risultati del **Task 2B** rispondono in modo netto alla domanda centrale del laboratorio (*"Do call order and transition structure matter for this dataset?"*).

**1. Analisi dei Risultati: Unigram (2A) vs Bigram (2B) per il tuo Report**

* **Esplosione del Vocabolario:** Passando da `ngram_range=(1, 1)` a `(1, 2)`, il numero di feature passa da **256** (i singoli unigrammi) a **5.741 colonne**, aggiungendo ben **5.485 transizioni adiacenti (bigrammi) distinte** osservate in `X_train`.


* **Balzo netto delle prestazioni (Validation Set):** A parità di modello (`LogisticRegression(C=1.0, class_weight=None)`), l'aggiunta dei bigrammi fa balzare la **Validation Macro-F1 da 0.7055 a 0.8508** (+14,5 punti percentuali!):


* Il guadagno avviene tutto sulla classe minoritaria **Goodware (`0`)**: la **Goodware Recall** raddoppia passando dal **31,15% al 64,75%**, e la **Goodware Precision** sale dal **67,86% al 79,00%** (portando la F1 sul Goodware da `0.4270` a `0.7117`), mentre la F1 sul Malware sale a `0.9899`.
* Anche nelle configurazioni con `class_weight="balanced"`, il modello Bigram recupera più di 20 punti di Macro-F1 rispetto all'Unigram (`0.8384` vs `0.6274` per $C=10.0$), perché nello spazio a 5.741 dimensioni le coppie di chiamate adiacenti permettono all'iperpiano lineare di separare il Goodware dal Malware con molti meno falsi positivi.




* **Perché le coppie di chiamate adiacenti aiutano così tanto? (Domanda Q2 del Task 2B):**
* Singole chiamate API di basso livello (`LdrLoadDll`, `RegOpenKeyExW`, `NtClose`) sono ubiquitarie in qualsiasi eseguibile Windows. Tuttavia, **la transizione immediata $A \rightarrow B$** cattura micro-operazioni semantiche (es. aprire una chiave di registro e modificarla subito dopo, oppure allocare un buffer e iniettarvi codice) che distinguono nettamente il flusso operativo di un malware da quello di un software benigno.





**2. Analisi dello Shuffle Test (Sezione 4.4 & Domanda Q3 del Task 2B)**

Il confronto sul Test Set prima e dopo lo shuffle è la prova regina da mettere in evidenza nel report:

* **Unigram BoW (2A):**
* Elementi della matrice cambiati dopo lo shuffle: **0**.


* Predizioni individuali cambiate: **0 su 6.505**.


* Test Macro-F1: resta identica al quarto decimale (**0.7115** prima $\rightarrow$ **0.7115** dopo). Questo conferma che la rappresentazione unigram dipende solo dalle frequenze marginali ed è totalmente invariante alle permutazioni.




* **Bigram BoW (2B):**
* Elementi della matrice cambiati dopo lo shuffle: **505.045**.


* Predizioni individuali cambiate: **1.308 su 6.505** (oltre il **20%** dei programmi del test set cambia classe predetta!).


* Test Macro-F1: sul test set originale il modello Bigram ottiene un eccellente **0.8572**, ma appena rimescoliamo l'ordine delle chiamate **crolla a 0.5026** (-35,5 punti percentuali, finendo quasi al livello del Dummy Classifier `0.4905` e molto *sotto* l'Unigram `0.7115`!).


* **Perché sul Test Shuffled il Bigram (`0.5026`) va persino peggio dell'Unigram (`0.7115`)?** Perché durante il training il modello Bigram ha assegnato pesi importanti ($\beta_j$) alle 5.485 feature di transizione $A \rightarrow B$. Quando rimescoli le sequenze a test time, distruggi i bigrammi reali e crei **bigrammi artificiali casuali** che attivano a sproposito i pesi del modello, ingannandolo attivamente (la Precision sul Goodware crolla dal 76,6% al 7,3%!).





**3. Baseline ufficiale selezionata per il resto del Lab (Task 2 $\rightarrow$ Task 6)**

Come richiesto a fine Task 2:

* **Miglior modello BoW (selezionato su Validation Set):** **Bigram BoW (`ngram_range=(1, 2)`) + Logistic Regression ($C=1.0$, No Weights)**.


* **Baseline Validation Macro-F1 da battere nei Task 3, 4, 5:** **0.8508**.


* **Test Score per la tabella finale del Task 6:** Test Originale = **0.8572**, Test Shuffled = **0.5026**, Training Time = **0.814s**.


# Task 3

# Task 4

## Risposte domande pre-training

- Servono ancora padding o truncation in train/val? E sul test set (dove le sequenze arrivano a 100)?

**Truncation: NO, mai**. A differenza della FFNN (che ha una matrice di pesi legata a una lunghezza fissa $L=90$), una RNN applica la stessa cella ricorrente con gli stessi pesi a ogni passo temporale $t = 1, \dots, T$. Può quindi processare senza problemi sequenze di lunghezza 60–90 in training e sequenze di lunghezza 70–100 a test time senza troncare alcuna chiamata API.

**Padding: SÌ, ma solo per motivi tecnici di mini-batching** (e va mascherato). In PyTorch, per elaborare un batch di es. 64 sequenze in parallelo su GPU, il tensore deve essere rettangolare (batch_size, max_len_in_batch). Applichiamo quindi un padding dinamico (solo fino alla sequenza più lunga di quel singolo batch) con il token <PAD> = 0, ma usiamo padding_idx=0 nell'embedding e pack_padded_sequence nella RNN: così la rete salta fisicamente i token di padding e non aggiorna lo stato nascosto dopo la fine reale di ciascuna sequenza.

- Confronto parametri allenabili (RNN vs FFNN) all'aumentare della lunghezza $L$

Nella FFNN del tuo collega (con embedding appiattiti), il primo layer lineare ha dimensione (L * embed_dim, hidden_dim): se raddoppi $L$, i parametri di quel layer raddoppiano.   Nella RNN, i pesi ricorrenti ($W_{ih}, W_{hh}$) sono condivisi tra tutti i passi temporali (weight sharing): il numero di parametri dipende solo da embed_dim e hidden_dim, ed è totalmente indipendente dalla lunghezza della sequenza $L$. (Quando il tuo collega finirà il Task 3, ti basterà confrontare il numero esatto stampato dal nostro script con il suo!).

- Bidirectional RNN: ha senso guardare le chiamate future per questo task?

Sì, in questo scenario (analisi post-esecuzione / offline): il dataset è raccolto eseguendo il programma in una Cuckoo Sandbox e classificando la traccia completa dopo che l'esecuzione è terminata. Conoscere anche il contesto successivo aiuta la rete a interpretare chiamate iniziali ambigue (es. capire se un OpenProcess iniziale porta poi a un'iniezione di codice).

---

- Limite pratico:
se invece volessimo costruire un sistema di rilevamento online / in tempo reale che blocca il malware mentre è ancora in esecuzione, una Bidirectional RNN non sarebbe utilizzabile perché non può accedere alle chiamate future.

In [ ]:
import time
import copy
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence

# =====================================================================
# 1. Costruzione Vocabolario SOLO su X_train (256 API + <PAD> + <UNK>)
# =====================================================================
PAD_IDX = 0
UNK_IDX = 1

vocab = {"<PAD>": PAD_IDX, "<UNK>": UNK_IDX}
for seq in X_train:
    for call in seq:
        if call not in vocab:
            vocab[call] = len(vocab)

VOCAB_SIZE = len(vocab)
print(f"Vocab size totale (inclusi <PAD> e <UNK>): {VOCAB_SIZE}")  # Atteso: 258

def encode_sequence(seq, vocab_dict):
    return [vocab_dict.get(call, UNK_IDX) for call in seq]

# =====================================================================
# 2. Dataset & DataLoader con Padding Dinamico (nessuna truncation!)
# =====================================================================
class APISequenceDataset(Dataset):
    def __init__(self, sequences, labels, vocab_dict):
        self.encoded = [torch.tensor(encode_sequence(s, vocab_dict), dtype=torch.long) for s in sequences]
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.encoded[idx], self.labels[idx]

def collate_fn(batch):
    sequences, labels = zip(*batch)
    lengths = torch.tensor([len(s) for s in sequences], dtype=torch.long)
    # Padding dinamico solo fino alla lunghezza massima di QUESTO batch
    padded_seqs = pad_sequence(sequences, batch_first=True, padding_value=PAD_IDX)
    labels = torch.stack(labels)
    return padded_seqs, lengths, labels

BATCH_SIZE = 64

train_ds = APISequenceDataset(X_train, y_train, vocab)
val_ds   = APISequenceDataset(X_val, y_val, vocab)
test_ds  = APISequenceDataset(X_test, y_test, vocab)
test_sh_ds = APISequenceDataset(X_test_sh, y_test, vocab)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)
test_loader  = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)
test_sh_loader = DataLoader(test_sh_ds, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

# =====================================================================
# 3. Architettura Unificata (Vanilla RNN, BiRNN, LSTM, BiLSTM)
# =====================================================================
class APIRNNClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim=64, hidden_dim=128,
                 rnn_type="RNN", bidirectional=False, num_layers=1, dropout=0.2):
        super().__init__()
        self.rnn_type = rnn_type
        self.bidirectional = bidirectional

        # padding_idx=0 mantiene il vettore di embedding del <PAD> sempre a zero
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=PAD_IDX)
        self.emb_dropout = nn.Dropout(dropout)

        rnn_cls = nn.LSTM if rnn_type == "LSTM" else nn.RNN
        self.rnn = rnn_cls(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=bidirectional,
            dropout=dropout if num_layers > 1 else 0.0
        )

        fc_in_dim = hidden_dim * 2 if bidirectional else hidden_dim
        self.fc = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(fc_in_dim, 2)  # 2 logits: Goodware (0), Malware (1)
        )

    def forward(self, x, lengths):
        emb = self.emb_dropout(self.embedding(x))
        # pack_padded_sequence impedisce alla RNN di processare i token <PAD>
        packed = pack_padded_sequence(emb, lengths.cpu(), batch_first=True, enforce_sorted=False)

        if self.rnn_type == "LSTM":
            _, (h_n, _) = self.rnn(packed)
        else:
            _, h_n = self.rnn(packed)

        # Estraiamo lo stato nascosto finale dell'ultimo layer
        if self.bidirectional:
            # Concatena stato finale forward (h_n[-2]) e backward (h_n[-1])
            feat = torch.cat((h_n[-2], h_n[-1]), dim=1)
        else:
            feat = h_n[-1]

        return self.fc(feat)

def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

In [ ]:
@torch.no_grad()
def predict_loader(model, loader, eval_device=device):
    model.eval()
    all_preds, all_labels = [], []
    total_loss = 0.0
    criterion_eval = nn.CrossEntropyLoss()

    for x, lengths, y in loader:
        x, y = x.to(eval_device), y.to(eval_device)
        logits = model(x, lengths)
        loss = criterion_eval(logits, y)
        total_loss += loss.item() * y.size(0)
        preds = logits.argmax(dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(y.cpu().numpy())

    avg_loss = total_loss / len(loader.dataset)
    return np.array(all_labels), np.array(all_preds), avg_loss

def train_rnn_model(model, train_loader, val_loader, epochs=15, lr=1e-3,
                    class_weights=None, clip_norm=1.0, run_device=device, verbose=True):
    set_seed(SEED)
    model = model.to(run_device)
    criterion = nn.CrossEntropyLoss(weight=class_weights.to(run_device) if class_weights is not None else None)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    history = {"train_loss": [], "val_loss": [], "val_macro_f1": [], "epoch_times": []}
    best_val_f1 = -1.0
    best_weights = copy.deepcopy(model.state_dict())

    for epoch in range(1, epochs + 1):
        t0 = time.time()
        model.train()
        running_loss = 0.0

        for x, lengths, y in train_loader:
            x, y = x.to(run_device), y.to(run_device)
            optimizer.zero_grad()
            logits = model(x, lengths)
            loss = criterion(logits, y)
            loss.backward()

            # Gradient clipping contro exploding gradients (come richiesto da traccia)
            if clip_norm is not None:
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=clip_norm)

            optimizer.step()
            running_loss += loss.item() * y.size(0)

        epoch_time = time.time() - t0
        train_loss = running_loss / len(train_loader.dataset)

        # Valutazione su Validation Set
        y_true_val, y_pred_val, val_loss = predict_loader(model, val_loader, eval_device=run_device)
        val_f1 = f1_score(y_true_val, y_pred_val, average="macro", zero_division=0)

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["val_macro_f1"].append(val_f1)
        history["epoch_times"].append(epoch_time)

        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_weights = copy.deepcopy(model.state_dict())

        if verbose:
            print(f"Epoch {epoch:02d}/{epochs} | Train Loss: {train_loss:.4f} | "
                  f"Val Loss: {val_loss:.4f} | Val Macro-F1: {val_f1:.4f} | Time: {epoch_time:.2f}s")

    model.load_state_dict(best_weights)
    return model, history, best_val_f1

def plot_training_curves(histories_dict):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    for name, hist in histories_dict.items():
        epochs_range = range(1, len(hist["train_loss"]) + 1)
        axes[0].plot(epochs_range, hist["train_loss"], linestyle="--", alpha=0.7, label=f"{name} (Train)")
        axes[0].plot(epochs_range, hist["val_loss"], linewidth=2, label=f"{name} (Val)")
        axes[1].plot(epochs_range, hist["val_macro_f1"], marker="o", markersize=4, label=name)

    axes[0].set_title("Training & Validation Loss")
    axes[0].set_xlabel("Epoca")
    axes[0].set_ylabel("Cross-Entropy Loss")
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    axes[1].set_title("Validation Macro-F1 per Epoca")
    axes[1].set_xlabel("Epoca")
    axes[1].set_ylabel("Macro-F1")
    axes[1].axhline(0.8508, color="black", linestyle=":", label="BoW Bigram Baseline (0.8508)")
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

In [ ]:
# =====================================================================
# TASK 4: Confronto delle 3 Varianti (Stessi iperparametri di base)
# =====================================================================
EPOCHS = 15
LR = 1e-3
EMBED_DIM = 64
HIDDEN_DIM = 128

# 1. Unidirectional Vanilla RNN
print("\n>>> RUN A: Unidirectional Vanilla RNN <<<")
model_rnn = APIRNNClassifier(VOCAB_SIZE, EMBED_DIM, HIDDEN_DIM, rnn_type="RNN", bidirectional=False)
print(f"Parametri allenabili (Vanilla RNN): {count_parameters(model_rnn):,}")
model_rnn, hist_rnn, best_f1_rnn = train_rnn_model(
    model_rnn, train_loader, val_loader, epochs=EPOCHS, lr=LR, class_weights=None
)
y_v, p_v, _ = predict_loader(model_rnn, val_loader)
evaluate_predictions(y_v, p_v, "Validation - Unidirectional Vanilla RNN")

# 2. Bidirectional Vanilla RNN
print("\n>>> RUN B: Bidirectional Vanilla RNN <<<")
model_birnn = APIRNNClassifier(VOCAB_SIZE, EMBED_DIM, HIDDEN_DIM, rnn_type="RNN", bidirectional=True)
print(f"Parametri allenabili (BiRNN): {count_parameters(model_birnn):,}")
model_birnn, hist_birnn, best_f1_birnn = train_rnn_model(
    model_birnn, train_loader, val_loader, epochs=EPOCHS, lr=LR, class_weights=None
)
y_v, p_v, _ = predict_loader(model_birnn, val_loader)
evaluate_predictions(y_v, p_v, "Validation - Bidirectional Vanilla RNN")

# 3. Bidirectional LSTM (scelta motivata: gestisce dipendenze lunghe + contesto bidirezionale offline)
print("\n>>> RUN C: Bidirectional LSTM <<<")
model_bilstm = APIRNNClassifier(VOCAB_SIZE, EMBED_DIM, HIDDEN_DIM, rnn_type="LSTM", bidirectional=True)
print(f"Parametri allenabili (BiLSTM): {count_parameters(model_bilstm):,}")
model_bilstm, hist_bilstm, best_f1_bilstm = train_rnn_model(
    model_bilstm, train_loader, val_loader, epochs=EPOCHS, lr=LR, class_weights=None
)
y_v, p_v, _ = predict_loader(model_bilstm, val_loader)
evaluate_predictions(y_v, p_v, "Validation - Bidirectional LSTM")

# Plot comparativo delle curve di training e validation
plot_training_curves({
    "Vanilla RNN": hist_rnn,
    "BiRNN": hist_birnn,
    "BiLSTM": hist_bilstm
})

In [ ]:
# =====================================================================
# 1. SALVATAGGIO IMMEDIATO DEL BEST MODEL (non perdere 70 min di CPU!)
# =====================================================================
torch.save(model_bilstm.state_dict(), "best_bilstm_task4.pt")
print("Modello BiLSTM salvato in 'best_bilstm_task4.pt'.")

# Riepilogo tempi medi per epoca (su CPU/device attuale)
avg_time_rnn    = np.mean(hist_rnn["epoch_times"])
avg_time_birnn  = np.mean(hist_birnn["epoch_times"])
avg_time_bilstm = np.mean(hist_bilstm["epoch_times"])
tot_time_bilstm = np.sum(hist_bilstm["epoch_times"])

print(f"\n--- Tempi Medi per Epoca ({device}) ---")
print(f"Vanilla RNN: {avg_time_rnn:.2f}s/epoca (Totale: {np.sum(hist_rnn['epoch_times']):.1f}s)")
print(f"BiRNN:       {avg_time_birnn:.2f}s/epoca (Totale: {np.sum(hist_birnn['epoch_times']):.1f}s)")
print(f"BiLSTM:      {avg_time_bilstm:.2f}s/epoca (Totale: {tot_time_bilstm:.1f}s)\n")

# =====================================================================
# 2. ORDER ABLATION (Shuffle Test sul Test Set) — Solo inferenza!
# =====================================================================
print("--- SHUFFLE TEST SUL TEST SET: Best RNN (BiLSTM) ---")
y_test_true, y_test_pred_bilstm, _ = predict_loader(model_bilstm, test_loader, eval_device=device)
y_test_sh_true, y_test_sh_pred_bilstm, _ = predict_loader(model_bilstm, test_sh_loader, eval_device=device)

test_bilstm_orig = evaluate_predictions(
    y_test_true, y_test_pred_bilstm,
    dataset_name="TEST ORIGINALE - Best RNN (BiLSTM)"
)
test_bilstm_shuf = evaluate_predictions(
    y_test_sh_true, y_test_sh_pred_bilstm,
    dataset_name="TEST SHUFFLED  - Best RNN (BiLSTM)"
)

pred_diff_bilstm = np.sum(y_test_pred_bilstm != y_test_sh_pred_bilstm)
print(f"Predizioni individuali cambiate dopo shuffle (BiLSTM): "
      f"{pred_diff_bilstm} / {len(y_test_true)} ({100*pred_diff_bilstm/len(y_test_true):.2f}%)\n")

# =====================================================================
# 3. Grafico Confronto Finale: BoW vs BiLSTM (Original vs Shuffled Test)
# =====================================================================
models_cmp = ["Unigram BoW (2A)", "Bigram BoW (2B)", "BiLSTM (Task 4)"]
orig_cmp = [test_uni_orig["macro_f1"], test_bi_orig["macro_f1"], test_bilstm_orig["macro_f1"]]
shuf_cmp = [test_uni_shuf["macro_f1"], test_bi_shuf["macro_f1"], test_bilstm_shuf["macro_f1"]]

x = np.arange(len(models_cmp))
width = 0.35

fig, ax = plt.subplots(figsize=(9, 5))
r1 = ax.bar(x - width/2, orig_cmp, width, label="Original Test Set", color="seagreen")
r2 = ax.bar(x + width/2, shuf_cmp, width, label="Shuffled Test Set", color="crimson")
ax.bar_label(r1, fmt="%.4f", padding=3, fontsize=9)
ax.bar_label(r2, fmt="%.4f", padding=3, fontsize=9)

ax.set_title("Order Ablation sul Test Set: BoW vs BiLSTM")
ax.set_xticks(x)
ax.set_xticklabels(models_cmp)
ax.set_ylabel("Test Macro-F1")
ax.set_ylim(0, 1.05)
ax.legend()
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

# Task 5

# Task 6